# Country vulnerability differences across leave-one-ot scenarios

This notebook calculates variable contribution to mean vulnerability for all leave-one-out scenarios using the different scenario vulnerability rasters. It compares the country mean vulnerability results and ranking of countries across different scenarios. It does so by creating scatterplots in a small-multiple comparing scenario and baseline country mean vulnerability ranks. It then visualizes the top countries with highest mean vulnerability across scenarios using stacked bar plots and small-multiple comparison figures. It creates:

- csv files for ten different scenarios showing variable contribution to mean vulnerability per country
- csv files for ten different scenarios showing the differences of mean vulnerability per country between baseline indicator and scenarios (rank and absolute difference)
- small-multiple with scatterplots of vulnerability ranks for the baseline indicator and each scenario
- small-multiple with stacked pillar diagrams showing 10 most vulnerable countries with variable contributions for each scenario

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `vulnerability_mean__*.tif` (all ten leave-one-out scenarios)
- `country_id.tiff`
- `country_id_lookup.csv`
- `country_variable_contrib.csv`
- `bii_5000m.tif`
- `wdpa_5000m.tif`
- `landmark_5000m.tif`
- `kba_5000m.tif`
- `poverty_5000m.tif`
- `water_risk_5000m.tif`
- `conflict_5000m.tif`
- `edi_5000m.tif` 
- `landrights_5000m.tif`
- `rule_of_law_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)


In [ ]:
#import packages
import pandas as pd
import geopandas as gpd
import os, math, glob
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio import warp
from rasterio import features
import geopandas as gpd
from shapely.geometry import box
from rasterio.features import geometry_mask
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from shapely.ops import unary_union
from rasterio.features import rasterize
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from shapely.geometry import box
from pathlib import Path
import glob
import os
import re
from adjustText import adjust_text

In [ ]:
# Configuration (edit these paths if needed)

#input paths
COUNTRY_ID = 'vulnerability_indicator\\countries\\country_id.tif'  
LOOK_UP = 'vulnerability_indicator\\countries\\country_id_lookup.csv' 
# Folder that contains all vulnerability_mean scenario rasters
vuln_scen_dir = 'vulnerability_indicator\\scenarios_missing_variable\\scenarios_one_missing'
vuln_mean_pattern = os.path.join(vuln_scen_dir, "vulnerability_mean__*.tif")


# Base variable rasters
sens_vars = {
    "Biodiversity Intactness":    'sensitivity\\biodiversity_intactness\\bii_5000m.tif',
    "Protected Areas":  'sensitivity\\protected_areas\\wdpa_5000m.tif',
    "IPLC Lands": 'sensitivity\\ind_com_lands\\landmark_5000m.tif' ,
    "Key Biodiversity Areas":'sensitivity\\kba\\kba_5000m.tif',
    "Poverty":   'sensitivity\\poverty\\poverty_5000m.tif',
    "Water Risk": 'sensitivity\\water_risk\\water_risk_5000m.tif',
}

lackof_adapt_vars= {
    "Conflict":  'lackof_adapt\\conflict\\conflict_5000m.tif',
    "Environmental Democracy": 'lackof_adapt\\environmental_democracy\\edi_5000m.tif',
    "Landrights":'lackof_adapt\\landrights\\landrights_5000m.tif',
    "Rule of Law":'lackof_adapt\\rule_of_law\\rule_of_law_5000m.tif'
}

#paths to different scenarios files
BASE_CON = 'vulnerability_indicator\\countries\\country_variable_contrib.csv'
SCEN_CONT = Path('vulnerability_indicator\\countries\\country_means_scenarios')
SCEN_DIFF=Path('vulnerability_indicator\\countries\\country_means_scenarios\\per_country_differences')

# Output paths
out_csv_dir = 'vulnerability_indicator\\countries\\country_means_scenarios'
os.makedirs(out_csv_dir, exist_ok=True)
out_diff_png= 'vulnerability_indicator\\countries\\country_means_scenarios\\per_country_differencesspearman_rank_comparison_small_multiples.png'


#other parameters
window_size = 2048
dst_nodata = -9999.0


In [ ]:
#Calculate variable contribtuion to mean vulnerability (different leave-one-out scenarios)

# Helper: map omitted filename stem -> variable key
def _stem(p):
    return os.path.splitext(os.path.basename(p))[0]

def find_key_by_stem(vars_dict, omitted_stem):
    """Return key in vars_dict whose path basename stem matches omitted_stem."""
    for k, p in vars_dict.items():
        if _stem(p) == omitted_stem:
            return k
    return None

def parse_omits_from_vuln_mean_filename(vuln_mean_path):
    name = _stem(vuln_mean_path)

    prefix = "vulnerability_mean__"
    if name.startswith(prefix):
        name = name[len(prefix):]

    parts = name.split("__A_")
    if len(parts) != 2:
        return (None, None)

    s_part, a_part = parts[0], parts[1]

    # s_part looks like: "S_all" or "S_omit_bii_5000m"
    sens_omit_stem = None
    if s_part.startswith("S_omit_"):
        sens_omit_stem = s_part[len("S_omit_"):]

    # a_part looks like: "all" or "omit_conflict_5000m"
    lack_omit_stem = None
    if a_part.startswith("all"):
        lack_omit_stem = None
    elif a_part.startswith("omit_"):
        lack_omit_stem = a_part[len("omit_"):]
    elif a_part.startswith("A_omit_"):
        # (just in case you ever include A_ in that part)
        lack_omit_stem = a_part[len("A_omit_"):]

    return (sens_omit_stem, lack_omit_stem)


#function to calculate contribution
def compute_country_variable_contrib(country_id_path, lookup_csv, vuln_mean_path, out_csv,
                                     sens_vars, lackof_adapt_vars):
    lookup = pd.read_csv(lookup_csv)
    max_id = int(lookup["country_id"].max())

    count = np.zeros(max_id + 1, dtype=np.int64)
    sum_v = np.zeros(max_id + 1, dtype=np.float64)

    sum_contrib = {f"sens_{k}": np.zeros(max_id + 1, dtype=np.float64) for k in sens_vars}
    sum_contrib.update({f"lack_{k}": np.zeros(max_id + 1, dtype=np.float64) for k in lackof_adapt_vars})

    sens_srcs = {k: rasterio.open(p) for k, p in sens_vars.items()}
    lack_srcs = {k: rasterio.open(p) for k, p in lackof_adapt_vars.items()}
    vsrc = rasterio.open(vuln_mean_path)
    csrc = rasterio.open(country_id_path)

    try:
        n_rows = math.ceil(vsrc.height / window_size)
        n_cols = math.ceil(vsrc.width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, vsrc.width - x_off)
                h = min(window_size, vsrc.height - y_off)
                window = Window(x_off, y_off, w, h)

                cid = csrc.read(1, window=window)
                v = vsrc.read(1, window=window)

                valid_v = (cid > 0) & np.isfinite(v) & (v != dst_nodata)
                if not np.any(valid_v):
                    continue

                cc = cid[valid_v].astype(np.int32)
                vv = v[valid_v].astype(np.float32)

                count += np.bincount(cc, minlength=max_id + 1)
                sum_v += np.bincount(cc, weights=vv.astype(np.float64), minlength=max_id + 1)

               # sensitivity vars
                sens_stack = []
                sens_valid_stack = []
                for k, src in sens_srcs.items():
                    a = src.read(1, window=window)
                    a_valid = np.isfinite(a) & (a != dst_nodata)
                    sens_stack.append(a)
                    sens_valid_stack.append(a_valid)

                kS = np.zeros_like(v, dtype=np.float32)
                for a_valid in sens_valid_stack:
                    kS[valid_v] += a_valid[valid_v].astype(np.float32)
                kS[valid_v] = np.maximum(kS[valid_v], 1.0)

                for (k, a, a_valid) in zip(sens_vars.keys(), sens_stack, sens_valid_stack):
                    use = valid_v & a_valid
                    if np.any(use):
                        contrib = 0.5 * (a[use].astype(np.float32) / kS[use])
                        sum_contrib[f"sens_{k}"] += np.bincount(
                            cid[use].astype(np.int32),
                            weights=contrib.astype(np.float64),
                            minlength=max_id + 1
                        )

                #  lack of adapt vars
                lack_stack = []
                lack_valid_stack = []
                for k, src in lack_srcs.items():
                    a = src.read(1, window=window)
                    a_valid = np.isfinite(a) & (a != dst_nodata)
                    lack_stack.append(a)
                    lack_valid_stack.append(a_valid)

                kL = np.zeros_like(v, dtype=np.float32)
                for a_valid in lack_valid_stack:
                    kL[valid_v] += a_valid[valid_v].astype(np.float32)
                kL[valid_v] = np.maximum(kL[valid_v], 1.0)

                for (k, a, a_valid) in zip(lackof_adapt_vars.keys(), lack_stack, lack_valid_stack):
                    use = valid_v & a_valid
                    if np.any(use):
                        contrib = 0.5 * (a[use].astype(np.float32) / kL[use])
                        sum_contrib[f"lack_{k}"] += np.bincount(
                            cid[use].astype(np.int32),
                            weights=contrib.astype(np.float64),
                            minlength=max_id + 1
                        )

        out = lookup.copy()
        out["pixel_count"] = out["country_id"].map(lambda i: int(count[int(i)]))
        out["mean_vulnerability"] = out["country_id"].map(
            lambda i: float(sum_v[int(i)] / count[int(i)]) if count[int(i)] > 0 else np.nan
        )

        for key in sum_contrib:
            out[f"contrib_{key}"] = out["country_id"].map(
                lambda i: float(sum_contrib[key][int(i)] / count[int(i)]) if count[int(i)] > 0 else np.nan
            )

        sens_cols = [f"contrib_sens_{k}" for k in sens_vars]
        lack_cols = [f"contrib_lack_{k}" for k in lackof_adapt_vars]

        out["contrib_sensitivity_total"] = out[sens_cols].sum(axis=1) if sens_cols else 0.0
        out["contrib_lack_total"] = out[lack_cols].sum(axis=1) if lack_cols else 0.0
        out["contrib_sum_vars"] = out["contrib_sensitivity_total"] + out["contrib_lack_total"]
        out["diff_vs_vuln"] = out["contrib_sum_vars"] - out["mean_vulnerability"]

        out.to_csv(out_csv, index=False)
        print(out_csv)

    finally:
        for src in sens_srcs.values():
            src.close()
        for src in lack_srcs.values():
            src.close()
        vsrc.close()
        csrc.close()


# Main loop: run for every vulnerability scenario
def run_country_means_for_all_vuln_scenarios():
    vuln_means = sorted(glob.glob(vuln_mean_pattern))
    if not vuln_means:
        raise RuntimeError(f"No vulnerability mean rasters found: {vuln_mean_pattern}")

    print(f"Found {len(vuln_means)} vulnerability scenarios")

    for vpath in vuln_means:
        sens_omit_stem, lack_omit_stem = parse_omits_from_vuln_mean_filename(vpath)

        # determine omitted keys from the master dictionaries
        sens_omit_key = find_key_by_stem(sens_vars, sens_omit_stem) if sens_omit_stem else None
        lack_omit_key = find_key_by_stem(lackof_adapt_vars, lack_omit_stem) if lack_omit_stem else None

        # create scenario-specific copies
        scenario_sens_vars = dict(sens_vars)
        scenario_lack_vars = dict(lackof_adapt_vars)

        if sens_omit_key and sens_omit_key in scenario_sens_vars:
            scenario_sens_vars.pop(sens_omit_key)

        if lack_omit_key and lack_omit_key in scenario_lack_vars:
            scenario_lack_vars.pop(lack_omit_key)

        scenario_name = _stem(vpath).replace("vulnerability_mean__", "")
        out_csv = os.path.join(out_csv_dir, f"country_means_{scenario_name}.csv")

        print("\nScenario:", scenario_name)
        if sens_omit_stem:
            print("  sensitivity omitted stem:", sens_omit_stem, "-> key:", sens_omit_key)
        if lack_omit_stem:
            print("  lack-of-adapt omitted stem:", lack_omit_stem, "-> key:", lack_omit_key)

        compute_country_variable_contrib(
            country_id_path=COUNTRY_ID,
            lookup_csv=LOOK_UP,
            vuln_mean_path=vpath,
            out_csv=out_csv,
            sens_vars=scenario_sens_vars,
            lackof_adapt_vars=scenario_lack_vars
        )

run_country_means_for_all_vuln_scenarios()


In [ ]:
#Compare different leave-one-out scenarios with baseline indicator (rank and absolute differences, spearman and pearson correlation)

#paths
baseline_path = BASE_CON
scenario_dir = SCEN_CONT


# 1) load baseline contribution file (all variables)
base = pd.read_csv(baseline_path)

# baseline needs: country_id + mean_vulnerability
base = base[["country_id", "iso", "name", "mean_vulnerability"]].copy()

# compute baseline rank 
base["rank_baseline"] = base["mean_vulnerability"].rank(ascending=False, method="min")

# 2) find scenario files
scenario_files = sorted(scenario_dir.glob("country_means_*.csv"))
print("Scenario files found:", len(scenario_files))
for f in scenario_files:
    print(" -", f.name)

# 3) FUNCTION: COMPARE ONE SCENARIO TO BASELINE
def compare_one_scenario(base_df, scenario_path):
    scen = pd.read_csv(scenario_path)
    scen = scen[["country_id", "mean_vulnerability"]].copy()
    scen = scen.rename(columns={"mean_vulnerability": "mean_vulnerability_scenario"})

    # rank scenario (1 = highest vulnerability)
    scen["rank_scenario"] = scen["mean_vulnerability_scenario"].rank(ascending=False, method="min")

    # merge baseline + scenario
    df = base_df.merge(scen, on="country_id", how="left")

    # calculate score change
    df["delta_v"] = df["mean_vulnerability_scenario"] - df["mean_vulnerability"]
    df["abs_delta_v"] = df["delta_v"].abs()

    #calculate rank change
    df["delta_rank"] = df["rank_scenario"] - df["rank_baseline"]
    df["abs_delta_rank"] = df["delta_rank"].abs()

    # pairwise valid rows for correlations
    valid_ranks = df[["rank_baseline", "rank_scenario"]].dropna()
    valid_scores = df[["mean_vulnerability", "mean_vulnerability_scenario"]].dropna()

    spearman_rho = float(valid_ranks.corr(method="spearman").iloc[0, 1]) if len(valid_ranks) >= 2 else np.nan
    pearson_r   = float(valid_scores.corr(method="pearson").iloc[0, 1])  if len(valid_scores) >= 2 else np.nan

    # global summary stats
    summary = {
        "scenario_file": scenario_path.name,
        "mean_abs_delta_v": float(df["abs_delta_v"].mean()),
        "max_abs_delta_v": float(df["abs_delta_v"].max()),
        "mean_abs_delta_rank": float(df["abs_delta_rank"].mean()),
        "max_abs_delta_rank": float(df["abs_delta_rank"].max()),
        "spearman_rho": spearman_rho,
        "pearson_r": pearson_r,
        "n_compared_ranks": int(len(valid_ranks)),
        "n_compared_scores": int(len(valid_scores)),
    }
        

    return df, summary

# 4) LOOP OVER ALL SCENARIOS
all_summaries = []
all_diffs = {}  # store per-country diffs per scenario (optional)

for scen_path in scenario_files:
    diff_df, summary = compare_one_scenario(base, scen_path)
    all_summaries.append(summary)
    all_diffs[scen_path.stem] = diff_df

# make summary table
summary_df = pd.DataFrame(all_summaries)

# sort: most robust (highest rho, lowest mean rank change)
summary_df = summary_df.sort_values(["spearman_rho", "mean_abs_delta_rank"], ascending=[False, True])


# 5) SAVE PER-COUNTRY DIFFS (one CSV per scenario)
out_diffs_dir = scenario_dir / "per_country_differences"
out_diffs_dir.mkdir(parents=True, exist_ok=True)

for scen_name, diff_df in all_diffs.items():
    out_path = out_diffs_dir / f"{scen_name}_diff_vs_baseline.csv"
    diff_df.to_csv(out_path, index=False)

print("Saved per-country diffs to:", out_diffs_dir)


# 6) SAVE SUMMARY TO CSV
out_summary_csv = scenario_dir / "robustness_summary_vs_baseline.csv"
summary_df.to_csv(out_summary_csv, index=False)
print("\nSaved summary to:", out_summary_csv)



In [ ]:
# Prepare scatterplots to compare ranks of leave-one-out scenarios and baseline indicator (4 steps)

#1) extract scenario name function

def extract_scenario_name(path):

    fname = os.path.basename(path)

    m = re.search(r"omit_([a-z_]+?)_5000m", fname)
    if m:
        return m.group(1)
    return "all"


In [ ]:
# 2) Prepare names for scatterplot

pretty_name = {
    "wdpa": "Protected Areas",
    "conflict": "Conflict",
    "edi": "Environmental Democracy",
    "landrights": "Land Rights",
    "rule_of_law": "Rule of Law",
    "bii": "Biodiversity Intactness",
    "kba": "Key Biodiversity Areas",
    "landmark": "IPLC Lands",
    "poverty": "Poverty",
    "water_risk": "Water Risk",
}


In [ ]:
# 3) Write function to create small multiples with scatterplots

def plot_rank_comparison_small_multiples(
    scenario_paths,
    outpath,
    ncols=3,
    top_n_highlight=3,
    top_n_labels=3,
):
    n = len(scenario_paths)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(5.5 * ncols, 5.5 * nrows),
        sharex=True,
        sharey=True,
        dpi=200,
    )

    axes = axes.flatten()

    for ax, scenario_path in zip(axes, scenario_paths):
        df = pd.read_csv(scenario_path)

        rho = df["rank_baseline"].corr(
            df["rank_scenario"], method="spearman"
        )

        top = df.sort_values(
            "abs_delta_rank", ascending=False
        ).head(top_n_highlight)

        # Background points
        ax.scatter(
            df["rank_baseline"],
            df["rank_scenario"],
            alpha=0.25,
            s=15,
        )

        # Highlighted points
        ax.scatter(
            top["rank_baseline"],
            top["rank_scenario"],
            alpha=1.0,
            s=25,
        )

        # Spearman rho annotation
        ax.text(
            0.97,
            0.03,
            f"ρ = {rho:.3f}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=10,
            fontweight="semibold",
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                edgecolor="none",
                alpha=0.85,
            ),
            zorder=5,
        )

        # Labels (limit to top movers)
        texts = []
        for _, r in top.head(top_n_labels).iterrows():
            texts.append(
                ax.text(
                    r["rank_baseline"],
                    r["rank_scenario"],
                    r["name"],
                    fontsize=12,
                )
            )

        adjust_text(
            texts,
            ax=ax,
            arrowprops=dict(arrowstyle="-", linewidth=0.5),
        )

        # Titles
        scen_name = extract_scenario_name(scenario_path)
        raw = extract_scenario_name(scenario_path)
        label = pretty_name.get(raw, raw)
        
        ax.set_title(
            f"Variable left out: {label}",
            fontsize=12,
            fontweight="semibold",
        )


        # Clean look
        ax.grid(True, alpha=0.2)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # Turn off unused axes
    for ax in axes[n:]:
        ax.axis("off")

    # Global labels
    fig.suptitle(
        "Country vulnerability rank comparison\nBaseline vs. scenario (Spearman correlation)",
        fontsize=16,
        fontweight="semibold",
        y=0.98,
    )

    for r in range(nrows):
        axes[r * ncols].set_ylabel("Scenario vulnerability rank", fontsize=12)

    for ax in axes[-ncols:]:
        ax.set_xlabel("Baseline vulnerability rank", fontsize=12)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(outpath, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


In [ ]:
# 4) Run functions, generate and save small multiple

scenario_dir = SCEN_DIFF


all_scenario_paths = sorted(
    glob.glob(os.path.join(scenario_dir, "country_means_*.csv"))
)

scenario_paths = [
    p for p in all_scenario_paths
    if extract_scenario_name(p) != "all"
]

outpath = out_diff_png

plot_rank_comparison_small_multiples(
    scenario_paths=scenario_paths,
    outpath=outpath,
    ncols=3,
    top_n_highlight=3,
    top_n_labels=3,
)


In [ ]:
# Create Small Multiple with stacked pillar diagrams for variable contributions for different leave-one-out scenarios

# Inputs
scenario_dir = SCEN_CONT
out_dir = scenario_dir / "plots_10"
out_dir.mkdir(parents=True, exist_ok=True)

expected_cols_to_proper = {}
for proper in sens_vars.keys():
    expected_cols_to_proper[f"contrib_sens_{proper}"] = proper
for proper in lackof_adapt_vars.keys():
    expected_cols_to_proper[f"contrib_lack_{proper}"] = proper


def get_vals_or_zeros(df, col, n):
    if col in df.columns:
        return df[col].to_numpy(dtype=np.float64)
    return np.zeros(n, dtype=np.float64)


def scenario_omitted_var(df):
    missing_cols = [c for c in expected_cols_to_proper if c not in df.columns]
    if len(missing_cols) == 1:
        return expected_cols_to_proper[missing_cols[0]]
    return "None"


def draw_one_panel(ax, df, scenario_name, sens_keys, lack_keys, sens_cols, lack_cols,
                   sens_colors, lack_colors, y_max=None, top_n=10):
    top = df.sort_values("mean_vulnerability", ascending=False).head(top_n).copy()

    x = np.arange(len(top))
    labels_x = top["name"].astype(str).tolist()

    bottom = np.zeros(len(top), dtype=np.float64)

    for col, color in zip(sens_cols, sens_colors):
        vals = get_vals_or_zeros(top, col, len(top))
        ax.bar(x, vals, bottom=bottom, color=color, edgecolor="white", linewidth=0.25)
        bottom += np.nan_to_num(vals, nan=0.0)

    for col, color in zip(lack_cols, lack_colors):
        vals = get_vals_or_zeros(top, col, len(top))
        ax.bar(x, vals, bottom=bottom, color=color, edgecolor="white", linewidth=0.25)
        bottom += np.nan_to_num(vals, nan=0.0)

    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#E3E6EA")
    ax.spines["bottom"].set_color("#E3E6EA")

    ax.set_xticks(x)
    ax.set_xticklabels(labels_x, rotation=60, ha="right", fontsize=12, color="#2B2F36")
    ax.tick_params(axis="y", labelsize=12, colors="#2B2F36")
    ax.margins(x=0.01)

    omitted = scenario_omitted_var(df)

    ax.set_title(
        f"Variable left out: {omitted}",
        fontsize=10,
        fontweight="semibold",
        pad=6,
        color="#2B2F36",
    )

    if y_max is None:
        y_max = float(np.nanmax(bottom)) * 1.05 if np.nanmax(bottom) > 0 else 1.0
    ax.set_ylim(0, y_max)

    return bottom



# Build file list-
csv_files_all = sorted(scenario_dir.glob("country_means_*.csv"))
if not csv_files_all:
    raise FileNotFoundError(f"No files matching country_means_*.csv in: {scenario_dir}")

scenario_names_all = [p.stem.replace("country_means_", "") for p in csv_files_all]


# Determine variable order / keys
sens_keys = list(sens_vars.keys())
lack_keys = list(lackof_adapt_vars.keys())

sens_cols = [f"contrib_sens_{k}" for k in sens_keys]
lack_cols = [f"contrib_lack_{k}" for k in lack_keys]

sens_cmap = plt.cm.inferno
lack_cmap = plt.cm.viridis
sens_colors = [sens_cmap(v) for v in np.linspace(0.15, 0.85, len(sens_cols))]
lack_colors = [lack_cmap(v) for v in np.linspace(0.15, 0.85, len(lack_cols))]

# drop baseline ("omitted == None")
csv_files = []
scenario_names = []
for p, scen in zip(csv_files_all, scenario_names_all):
    df_tmp = pd.read_csv(p)
    if scenario_omitted_var(df_tmp) == "None":
        continue  # baseline -> skip
    csv_files.append(p)
    scenario_names.append(scen)

if not csv_files:
    raise RuntimeError("After removing the baseline scenario, no scenario CSVs remain.")

# consistent y-limit across panels
use_global_ymax = True
global_ymax = None
if use_global_ymax:
    max_heights = []
    for p in csv_files:
        df = pd.read_csv(p)
        top = df.sort_values("mean_vulnerability", ascending=False).head(10).copy()
        bottom = np.zeros(len(top), dtype=np.float64)

        for col in sens_cols:
            bottom += np.nan_to_num(get_vals_or_zeros(top, col, len(top)), nan=0.0)
        for col in lack_cols:
            bottom += np.nan_to_num(get_vals_or_zeros(top, col, len(top)), nan=0.0)

        max_heights.append(float(np.nanmax(bottom)) if len(bottom) else 0.0)

    global_ymax = (max(max_heights) * 1.05) if max_heights else 1.0


# Layout: small multiples grid
n = len(csv_files)
ncols = 3 if n >= 3 else n
nrows = math.ceil(n / ncols)

fig_w = 6.2 * ncols
fig_h = 4.2 * nrows

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(fig_w, fig_h), dpi=200)
fig.patch.set_facecolor("white")
axes = np.array(axes).reshape(-1)


# Draw each panel
for ax, csv_path, scen in zip(axes, csv_files, scenario_names):
    df = pd.read_csv(csv_path)
    draw_one_panel(
        ax=ax,
        df=df,
        scenario_name=scen,         
        sens_keys=sens_keys,
        lack_keys=lack_keys,
        sens_cols=sens_cols,
        lack_cols=lack_cols,
        sens_colors=sens_colors,
        lack_colors=lack_colors,
        y_max=global_ymax,
        top_n=10
    )

for ax in axes[len(csv_files):]:
    ax.axis("off")

fig.suptitle(
    "Top 10 countries with highest mean vulnerability\nStacked contributions by variable",
    fontsize=16,
    fontweight="semibold",
    y=0.995,
    color="#2B2F36",
)

for r in range(nrows):
    left_ax = axes[r * ncols]
    left_ax.set_ylabel("Contribution to mean vulnerability", fontsize=10, color="#2B2F36")

# Shared legend
handles, labels = [], []
handles.append(Line2D([], [], linestyle="none"))
labels.append(r"$\bf{Sensitivity}$")
for k, color in zip(sens_keys, sens_colors):
    handles.append(Line2D([0], [0], color=color, lw=6))
    labels.append(k)

handles.append(Line2D([], [], linestyle="none"))
labels.append("")
handles.append(Line2D([], [], linestyle="none"))
labels.append(r"$\bf{Lack\ of\ adaptive\ capacity}$")
for k, color in zip(lack_keys, lack_colors):
    handles.append(Line2D([0], [0], color=color, lw=6))
    labels.append(k)

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.84, 0.5),
    frameon=False,
    fontsize=10,
    labelcolor="#2B2F36",
)

plt.tight_layout(rect=[0.0, 0.0, 0.82, 0.96])

# Save
out_png = out_dir / "countries10_mean__small_multiples.png"
fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")

out_pdf = out_dir / "countries10_mean__small_multiples.pdf"
fig.savefig(out_pdf, bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)

print(f"Saved:\n- {out_png}\n- {out_pdf}")
